In [1]:
import pandas as pd


In [ ]:
#pandas-dev__pandas-38248

In [2]:
base = pd.date_range("2000-01-01", periods=10, freq="s")

In [3]:
base

DatetimeIndex(['2000-01-01 00:00:00', '2000-01-01 00:00:01',
               '2000-01-01 00:00:02', '2000-01-01 00:00:03',
               '2000-01-01 00:00:04', '2000-01-01 00:00:05',
               '2000-01-01 00:00:06', '2000-01-01 00:00:07',
               '2000-01-01 00:00:08', '2000-01-01 00:00:09'],
              dtype='datetime64[ns]', freq='s')

In [4]:
help(pd.date_range)

Help on function date_range in module pandas.core.indexes.datetimes:

date_range(start=None, end=None, periods=None, freq=None, tz=None, normalize: 'bool' = False, name: 'Hashable | None' = None, inclusive: 'IntervalClosedType' = 'both', *, unit: 'str | None' = None, **kwargs) -> 'DatetimeIndex'
    Return a fixed frequency DatetimeIndex.

    Returns the range of equally spaced time points (where the difference between any
    two adjacent points is specified by the given frequency) such that they all
    satisfy `start <[=] x <[=] end`, where the first one and the last one are, resp.,
    the first and last time points in that range that fall on the boundary of ``freq``
    (if given as a frequency string) or that are valid for ``freq`` (if given as a
    :class:`pandas.tseries.offsets.DateOffset`). (If exactly one of ``start``,
    ``end``, or ``freq`` is *not* specified, this missing parameter can be computed
    given ``periods``, the number of timesteps in the range. See the note

In [6]:
head = pd.date_range("2000-01-01 00:00:01",periods=10,freq="s")
base < head

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True])

In [7]:
#astropy__astropy-10814

In [10]:
from astropy.coordinates import SkyCoord, CIRS
import numpy as np
import astropy.units as u
from astropy.time import Time


t = Time('2020-10-01T20:00') + np.random.uniform(0, 1, size=100000) * u.min
icrs = SkyCoord.from_name('Crab')
len(t),icrs,icrs.transform_to(CIRS(obstime=t))

(100000,
 <SkyCoord (ICRS): (ra, dec) in deg
     (83.6324, 22.0174)>,
 <SkyCoord (CIRS: obstime=['2020-10-01T20:00:32.850' '2020-10-01T20:00:51.642'
  '2020-10-01T20:00:06.051' ... '2020-10-01T20:00:34.868'
  '2020-10-01T20:00:23.840' '2020-10-01T20:00:52.446'], location=(0., 0., 0.) km): (ra, dec) in deg
     [(83.6796977 , 22.03035645), (83.67969772, 22.03035645),
      (83.67969766, 22.03035645), ..., (83.6796977 , 22.03035645),
      (83.67969769, 22.03035645), (83.67969772, 22.03035645)]>)

In [ ]:
import timeit
import statistics

from astropy.coordinates import SkyCoord, CIRS
import numpy as np
import astropy.units as u
from astropy.time import Time

def setup():
    global t, icrs 
    t = Time('2020-10-01T20:00') + np.random.uniform(0, 1, size=100000) * u.min
    icrs = SkyCoord.from_name('Crab')

def workload():
    icrs.transform_to(CIRS(obstime=t))#坐标系统转换

runtimes = timeit.repeat(workload, number=1, repeat=3, setup=setup)

# Print runtime mean and std deviation.
print("Mean:", statistics.mean(runtimes))
print("Std Dev:", statistics.stdev(runtimes))

In [ ]:
import timeit
import statistics
import astropy.units as u
    
def workload():
    u.Unit('km/s', format='fits')

runtimes = timeit.repeat(workload, number=1, repeat=2000)

print("Mean:", statistics.mean(runtimes[-1000:]))
print("Std Dev:", statistics.stdev(runtimes[-1000:]))



In [17]:
import astropy.units as u
x = u.Unit('km/s', format='fits')
type(x)


astropy.units.core.CompositeUnit

In [19]:
import timeit
import statistics
import astropy.units as u

def setup():
    # 创建大量不同的FITS单位字符串
    # 包括各种速度单位的表示法，以及一些复合单位
    # 增加数据量到400个不同的单位字符串
    global unit_strings
    base_units = [
        'km/s', 'm/s', 'km s-1', 'm s-1', 'km.s-1', 'm.s-1',
        'km s**-1', 'm s**-1', 'km / s', 'm / s', 'km s⁻¹', 'm s⁻¹',
        'km*s^-1', 'm*s^-1', 'km * s^-1', 'm * s^-1', 'km s^-1', 'm s^-1',
        'km.s**-1', 'm.s**-1', 'km/h', 'm/h', 'km hr-1', 'm hr-1',
        'km.hr-1', 'm.hr-1', 'km hr**-1', 'm hr**-1', 'km / hr', 'm / hr',
        'km*hr^-1', 'm*hr^-1', 'km hr^-1', 'm hr^-1', 'km.hr**-1', 'm.hr**-1',
        'km/min', 'm/min', 'km min-1', 'm min-1', 'km.min-1', 'm.min-1',
        'km min**-1', 'm min**-1', 'km / min', 'm / min', 'km*min^-1', 'm*min^-1',
        'km min^-1', 'm min^-1', 'km.min**-1', 'm.min**-1'
    ]
    
    # 创建复合单位：速度单位与其他单位的组合
    # 这些应该仍然可以被解析为速度单位（通过等效性检查）
    compound_units = []
    for base in ['km/s', 'm/s', 'km s-1', 'm s-1']:
        compound_units.extend([
            f'{base} * Hz',  # 速度 * 频率 = 加速度？
            f'{base} / Hz',  # 速度 / 频率 = 距离？
            f'{base} * s',   # 速度 * 时间 = 距离
            f'{base} / s',   # 速度 / 时间 = 加速度
            f'{base} * m',   # 速度 * 距离 = 面积/时间？
            f'{base} / m',   # 速度 / 距离 = 频率
            f'{base} * kg',  # 速度 * 质量 = 动量
            f'{base} / kg',  # 速度 / 质量 = 特定动量
        ])
    
    # 添加一些更复杂的表达式（但仍然有效）
    complex_units = []
    for i in range(50):
        complex_units.append(f'km/s * (s^{i%3+1}) / (m^{i%2+1}) * (kg^{i%4})')
    
    # 合并所有单位字符串
    unit_strings = base_units * 5 + compound_units * 3 + complex_units
    
    # 设置全局索引
    global current_index
    current_index = 0

def workload():
    # 保持原始功能：创建'km/s'单位
    # 但增加负载：每次调用解析多个单位字符串
    global current_index
    
    # 每次解析10个单位字符串（增加workload复杂度）
    for i in range(10):
        idx = (current_index + i) % len(unit_strings)
        unit_str = unit_strings[idx]
        
        # 使用FITS格式创建单位（原始功能的核心）
        try:
            print(unit_str)
            unit = u.Unit(unit_str, format='fits')
            
            # 检查是否与速度单位等价（增加一些计算）
            _ = unit.is_equivalent('km/s')
        except (ValueError, u.UnitsError):
            # 如果解析失败，跳过（某些复杂表达式可能无效）
            pass
    
    # 更新索引供下次调用使用
    current_index = (current_index + 10) % len(unit_strings)
    
    # 仍然执行原始操作以确保功能等价
    base_unit = u.Unit('km/s', format='fits')
setup()
workload()


km/s
m/s
km s-1
m s-1
km.s-1
m.s-1
km s**-1
m s**-1
km / s
m / s


In [11]:
import timeit
import statistics
from astropy.time import Time, TimeDelta
import astropy.units as u
import numpy as np

def setup():
    global t0, t1, dt
    dt = TimeDelta(1 * u.hour)
    t0 = Time('2021-01-01')
    t1 = Time('2022-01-01')
    
def workload():
    global t0, t1, dt
    Time(np.arange(t0, t1, dt))


In [12]:
dt = TimeDelta(1 * u.hour)# “生成一整年的小时级时间序列”
t0 = Time('2021-01-01')
t1 = Time('2022-01-01')
dt,t0,t1,Time(np.arange(t0, t1, dt))

(<TimeDelta object: scale='None' format='jd' value=0.041666666666666664>,
 <Time object: scale='utc' format='iso' value=2021-01-01 00:00:00.000>,
 <Time object: scale='utc' format='iso' value=2022-01-01 00:00:00.000>,
 <Time object: scale='utc' format='iso' value=['2021-01-01 00:00:00.000' '2021-01-01 01:00:00.000'
  '2021-01-01 02:00:00.000' ... '2021-12-31 21:00:00.000'
  '2021-12-31 22:00:00.000' '2021-12-31 23:00:00.000']>)

In [13]:
import timeit
import statistics
import astropy.io.fits
import astropy.units as u
from astropy.coordinates import (
    EarthLocation, AltAz, SkyCoord, CoordinateAttribute, BaseCoordinateFrame,
    UnitSphericalRepresentation, RepresentationMapping,
)
from astropy.time import Time

def setup():
    global ExampleFrame, coord_attr, coord
    class ExampleFrame(BaseCoordinateFrame):
        frame_specific_representation_info = {
            UnitSphericalRepresentation: [
                RepresentationMapping("lon", "fov_lon"),
                RepresentationMapping("lat", "fov_lat"),
            ]
        }
        default_representation = UnitSphericalRepresentation
        coord_attr = CoordinateAttribute(default=None, frame=AltAz)

    loc = EarthLocation.of_site("Roque de los Muchachos")
    t = Time.now()
    frame = AltAz(location=loc, obstime=t)
    coord = SkyCoord(0 * u.deg, 0 * u.deg, frame=frame)

def workload():
    global ExampleFrame, coord
    ExampleFrame(coord_attr=coord)

runtimes = timeit.repeat(workload, number=10, repeat=2000, setup=setup)

print("Mean:", statistics.mean(runtimes[-1000:]))
print("Std Dev:", statistics.stdev(runtimes[-1000:]))

Mean: 7.428549160249531e-05
Std Dev: 2.6567187901920085e-06


In [14]:
import numpy as np

def setup():
    global ExampleFrame, coord

    class ExampleFrame(BaseCoordinateFrame):
        frame_specific_representation_info = {
            UnitSphericalRepresentation: [
                RepresentationMapping("lon", "fov_lon"),
                RepresentationMapping("lat", "fov_lat"),
            ]
        }
        default_representation = UnitSphericalRepresentation
        coord_attr = CoordinateAttribute(default=None, frame=AltAz)

    loc = EarthLocation.of_site("Roque de los Muchachos")
    t = Time.now()
    frame = AltAz(location=loc, obstime=t)

    n = 10000
    lon = np.linspace(0, 1, n) * u.deg
    lat = np.linspace(0, 1, n) * u.deg
    coord = SkyCoord(lon, lat, frame=frame)
def workload():
    global ExampleFrame, coord
    ExampleFrame(coord_attr=coord)

runtimes = timeit.repeat(workload, number=10, repeat=2000, setup=setup)

print("Mean:", statistics.mean(runtimes[-1000:]))
print("Std Dev:", statistics.stdev(runtimes[-1000:]))

Mean: 7.778725074604153e-05
Std Dev: 2.8522908497440765e-06
